# ML Practical 4 — Sales Data Clustering using K-Means

**Objective:** Apply K-Means clustering on sales data and determine the optimal number of clusters using the Elbow Method.  
**Dataset:** sales_data_sample.csv — place the file downloaded from Kaggle in this folder.  
**Source:** https://www.kaggle.com/datasets/kyanyoga/sample-sales-data


---
## Step 1 — Import Libraries

We import:
- `pandas` and `numpy` for data handling
- `KMeans` from `sklearn.cluster` for clustering
- `StandardScaler` to normalize features before clustering (K-Means is distance-based)
- `matplotlib` and `seaborn` for visualization
- `silhouette_score` to quantitatively validate the quality of clusters

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score

import warnings
warnings.filterwarnings('ignore')

print('Libraries imported successfully')

---
## Step 2 — Load the Dataset

The CSV file uses Latin-1 encoding (not UTF-8) because some product names contain special characters.  
`shape` tells us the number of rows (orders) and columns.  
`head()` gives a preview of the data.

In [ ]:
# Place sales_data_sample.csv (from Kaggle) in this folder
df = pd.read_csv('sales_data_sample.csv', encoding='latin1')

print('Shape:', df.shape)
df.head()

---
## Step 3 — Explore the Dataset

`info()` shows column names, data types, and non-null counts.  
`describe()` gives statistics like mean, min, max for numerical columns.  
We also check for missing values that need to be handled before clustering.

In [ ]:
df.info()

In [ ]:
print('Missing values per column:')
print(df.isnull().sum()[df.isnull().sum() > 0])
df.describe()

---
## Step 4 — Select and Prepare Features for Clustering

K-Means only works on numerical data.  
We select the most business-relevant numerical features for customer/sales segmentation:
- `QUANTITYORDERED` — how much was ordered
- `PRICEEACH` — price per unit
- `SALES` — total sales value of the order

We drop rows with any missing values in these columns to keep the data clean.

In [ ]:
features = ['QUANTITYORDERED', 'PRICEEACH', 'SALES']

data = df[features].dropna()

print('Rows after cleaning:', data.shape[0])
data.describe()

---
## Step 5 — Normalize the Data

K-Means uses Euclidean distance. Without normalization, `SALES` (which can be in thousands) would completely dominate over `QUANTITYORDERED` (single digits).  
`StandardScaler` scales all features to mean=0, std=1 so each feature contributes equally to distance calculations.

In [ ]:
scaler     = StandardScaler()
data_scaled = scaler.fit_transform(data)

print('Data normalized. Shape:', data_scaled.shape)

---
## Step 6 — Elbow Method to Find Optimal K

The Elbow Method runs K-Means for different values of K (1 to 10) and records the **WCSS** (Within-Cluster Sum of Squares) — the total squared distance of each point from its cluster center.  

- As K increases, WCSS always decreases (more clusters = closer to each point).
- We look for the **"elbow"** — the point where adding more clusters gives diminishing returns.
- That K value is the optimal number of clusters.

In [ ]:
wcss = []
K_range = range(1, 11)

for k in K_range:
    kmeans = KMeans(n_clusters=k, init='k-means++', n_init=10, random_state=42)
    kmeans.fit(data_scaled)
    wcss.append(kmeans.inertia_)   # inertia_ = WCSS

# Plot the Elbow Curve
plt.figure(figsize=(9, 5))
plt.plot(K_range, wcss, marker='o', color='steelblue', linewidth=2)
plt.title('Elbow Method — Optimal Number of Clusters')
plt.xlabel('Number of Clusters (K)')
plt.ylabel('WCSS (Within-Cluster Sum of Squares)')
plt.xticks(K_range)
plt.grid(True, linestyle='--', alpha=0.6)
plt.show()

---
## Step 7 — Silhouette Score for Validation

The silhouette score is a second method to confirm the optimal K.  
It measures how well each point fits within its own cluster vs how far it is from neighboring clusters.  
Score ranges from -1 to +1 — **higher is better**. The K with the highest silhouette score is the best.

In [ ]:
sil_scores = []

for k in range(2, 11):   # silhouette needs at least 2 clusters
    kmeans = KMeans(n_clusters=k, init='k-means++', n_init=10, random_state=42)
    labels = kmeans.fit_predict(data_scaled)
    sil_scores.append(silhouette_score(data_scaled, labels))

best_k = np.argmax(sil_scores) + 2   # +2 because we started from k=2
print(f'Best K by Silhouette Score: {best_k}  (score = {max(sil_scores):.4f})')

plt.figure(figsize=(9, 5))
plt.plot(range(2, 11), sil_scores, marker='o', color='seagreen', linewidth=2)
plt.axvline(x=best_k, color='red', linestyle='--', label=f'Best K={best_k}')
plt.title('Silhouette Score vs Number of Clusters')
plt.xlabel('Number of Clusters (K)')
plt.ylabel('Silhouette Score')
plt.xticks(range(2, 11))
plt.legend()
plt.grid(True, linestyle='--', alpha=0.6)
plt.show()

---
## Step 8 — Train Final K-Means Model

We train K-Means with the optimal K identified from the elbow and silhouette analysis.  
`init='k-means++'` is a smarter initialization strategy — it spreads initial centroids apart to avoid poor clustering results.  
`fit_predict()` trains the model and returns the cluster label for each data point.

In [ ]:
# Set K based on elbow + silhouette analysis (adjust if needed)
optimal_k = best_k

kmeans_final = KMeans(n_clusters=optimal_k, init='k-means++', n_init=10, random_state=42)
data['Cluster'] = kmeans_final.fit_predict(data_scaled)

print(f'K-Means trained with K={optimal_k}')
print('\nCluster counts:')
print(data['Cluster'].value_counts().sort_index())

---
## Step 9 — Visualize the Clusters

We plot **SALES vs QUANTITYORDERED**, coloring each point by its cluster.  
The red 'X' marks show the cluster **centroids** — the center point of each cluster.  
This visualization helps us understand what each cluster represents (e.g., high-value bulk orders vs small low-value orders).

In [ ]:
palette = sns.color_palette('tab10', optimal_k)

plt.figure(figsize=(10, 6))
for cluster_id in range(optimal_k):
    subset = data[data['Cluster'] == cluster_id]
    plt.scatter(subset['QUANTITYORDERED'], subset['SALES'],
                label=f'Cluster {cluster_id}', alpha=0.6, s=40)

# Plot centroids (inverse-transform scaled centroids back to original scale)
centroids_original = scaler.inverse_transform(kmeans_final.cluster_centers_)
plt.scatter(centroids_original[:, 0], centroids_original[:, 2],
            s=200, c='red', marker='X', label='Centroids', zorder=5)

plt.title(f'K-Means Clustering (K={optimal_k}) — Sales vs Quantity Ordered')
plt.xlabel('Quantity Ordered')
plt.ylabel('Sales')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.4)
plt.show()

---
## Step 10 — Cluster Profile Analysis

We compute the **mean values of each feature per cluster** to understand what makes each cluster distinct.  
For example:
- Cluster with high SALES + high QUANTITYORDERED → bulk high-value customers
- Cluster with low SALES + low QUANTITYORDERED → small occasional buyers

In [ ]:
cluster_profile = data.groupby('Cluster')[features].mean().round(2)
print('Cluster Profiles (Mean values per cluster):')
print(cluster_profile)

In [ ]:
# Bar chart of average SALES per cluster
cluster_profile['SALES'].plot(kind='bar', color=sns.color_palette('tab10', optimal_k),
                               figsize=(8, 4), edgecolor='black')
plt.title('Average Sales per Cluster')
plt.xlabel('Cluster')
plt.ylabel('Average Sales')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

---
## Step 11 — Pairplot of All Clusters

A pairplot shows scatter plots for every combination of features, colored by cluster.  
This gives a complete picture of how well the clusters separate across all dimensions.

In [ ]:
sns.pairplot(data, hue='Cluster', vars=features,
             palette='tab10', diag_kind='kde', plot_kws={'alpha': 0.5})
plt.suptitle('Pairplot of Sales Features by Cluster', y=1.02)
plt.show()

---
## Summary

| Step | What we did |
|------|-------------|
| 1 | Imported libraries |
| 2 | Loaded sales_data_sample.csv |
| 3 | Explored data types, missing values, and statistics |
| 4 | Selected numerical features: QUANTITYORDERED, PRICEEACH, SALES |
| 5 | Normalized with StandardScaler |
| 6 | Applied Elbow Method (K=1 to 10), plotted WCSS curve |
| 7 | Used Silhouette Score to confirm optimal K |
| 8 | Trained final K-Means with optimal K |
| 9 | Visualized clusters with centroids |
| 10 | Analyzed cluster profiles (mean feature values per cluster) |
| 11 | Pairplot across all features |

**Why K-Means + Elbow Method?**
- K-Means is fast, scalable, and easy to interpret.
- The Elbow Method avoids guessing K — it uses the data to find where adding more clusters stops being useful.
- Silhouette Score provides a second, quantitative confirmation of the best K.
